# 01 · Qwen3-VL 本地实践：问答、OCR、Grounding

**硬件**：🟡 建议 12GB+ VRAM（默认 4B-Instruct，bf16 约 9GB；显存不足可换 2B 或加 4bit 量化）

## 本 notebook 你将学到

1. 跑通开源旗舰 VLM 系列的小尺寸模型：图像问答、多图对比
2. OCR / 文档理解：动态分辨率带来的能力（对照 [theory.md](../theory.md) 第 2 节）
3. **Grounding**：让模型输出 bounding box 并画出来——这是 GUI agent 的基石能力
4. 观察视觉 token 数量与图像分辨率的关系（成本直觉）

> 版本提示（2026-08）：Qwen3-VL 需要较新的 transformers（>=4.57）。模型卡以 [Qwen3-VL GitHub](https://github.com/QwenLM/Qwen3-VL) 为准。

In [ ]:
%pip install -q "transformers>=4.57" accelerate torch pillow requests matplotlib
# 显存紧张时取消注释: %pip install -q bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"   # 显存不足换 "Qwen/Qwen3-VL-2B-Instruct"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    # 4bit 量化（约省一半显存）:
    # quantization_config=__import__('transformers').BitsAndBytesConfig(load_in_4bit=True),
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"loaded {MODEL_ID} on {model.device}")

In [ ]:
# 封装一个对话函数，后面反复用
def chat(image_or_images, prompt, max_new_tokens=512):
    imgs = image_or_images if isinstance(image_or_images, list) else [image_or_images]
    content = [{"type": "image", "image": im} for im in imgs]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    n_visual = (inputs["input_ids"] == model.config.image_token_id).sum().item()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    reply = processor.batch_decode(
        out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return reply, n_visual

## 1. 图像问答 + 视觉 token 成本直觉

注意输出里的 `visual tokens` 数字：Qwen3-VL 按原生分辨率切 patch（并做 2x2 合并压缩），**图越大，token 越多，越贵**。

In [ ]:
import requests
from io import BytesIO
from PIL import Image

def load(url):
    return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

img_cats = load("http://images.cocodataset.org/val2017/000000039769.jpg")

reply, n_vis = chat(img_cats, "描述这张图。图里有几只猫？它们分别是什么姿势？")
print(f"[visual tokens: {n_vis}]\n{reply}")

# 同一张图缩小一半再问，对比 token 数
small = img_cats.resize((img_cats.width // 2, img_cats.height // 2))
_, n_vis_small = chat(small, "图里有几只猫？")
print(f"\n原尺寸 {img_cats.size}: {n_vis} tokens | 半尺寸 {small.size}: {n_vis_small} tokens")

对比 00 章的 CLIP 实验：CLIP 数不清猫，而 VLM 可以——因为 LLM 在视觉特征之上做了真正的推理。

## 2. OCR 与文档理解

换一张带文字的图（发票/截图/文档照片都行）。要求结构化输出是实战关键——**JSON 格式的抽取比自由文本转写更有用**。

In [ ]:
# 用一张维基百科的文档类图片做演示；换成你自己的发票/截图效果更直观
img_doc = load("https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Receipt_in_Costa_Rica.jpg/640px-Receipt_in_Costa_Rica.jpg")

reply, n_vis = chat(
    img_doc,
    "这是一张小票。请以 JSON 输出：{\"merchant\": 商家名, \"date\": 日期, \"total\": 总金额, \"items\": [商品列表]}。读不清的字段填 null，不要编造。",
)
print(f"[visual tokens: {n_vis}]\n{reply}")

**防幻觉要点**：提示里写明"读不清填 null，不要编造"。VLM 对模糊文字会自信地编——批量生产场景请用 02 章的专用 OCR 模型（带 grounding 可溯源）。

## 3. Grounding：输出并绘制 bounding box

Grounding = 把语言指代对应到像素坐标。Qwen3-VL 的检测输出习惯用 **0–1000 归一化坐标**。这项能力是 GUI agent（"点击登录按钮"）和机器人抓取的基础。

In [ ]:
import json, re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

reply, _ = chat(
    img_cats,
    '定位图中每只猫，以 JSON 数组输出: [{"label": "cat", "bbox_2d": [x1, y1, x2, y2]}]，坐标用 0-1000 归一化。',
)
print(reply)

m = re.search(r"\[.*\]", reply, re.S)
boxes = json.loads(m.group()) if m else []

W, H = img_cats.size
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_cats)
for b in boxes:
    x1, y1, x2, y2 = b["bbox_2d"]
    # 0-1000 归一化 -> 像素；若模型输出的是像素坐标则无需换算
    if max(x1, y1, x2, y2) <= 1000 and (x2 <= 1000):
        x1, x2 = x1 / 1000 * W, x2 / 1000 * W
        y1, y2 = y1 / 1000 * H, y2 / 1000 * H
    ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, color="red", lw=2))
    ax.text(x1, y1 - 5, b["label"], color="red", fontsize=10)
ax.axis("off")
plt.show()

## 4. 多图对比

现代 VLM 支持交错多图输入——这是"找不同"、商品对比、前后对照类应用的基础。

In [ ]:
img_bear = load("http://images.cocodataset.org/val2017/000000000285.jpg")

reply, n_vis = chat(
    [img_cats, img_bear],
    "对比这两张图：分别是什么动物？环境有什么不同？哪张更适合做壁纸，为什么？",
)
print(f"[visual tokens: {n_vis}]\n{reply}")

## 练习

1. 截一张你手机的设置页截图，让模型定位"无线局域网"入口的 bbox——体验 GUI grounding 精度。
2. 拍一张手写笔记，对比本模型与 02 章 DeepSeek-OCR 类专用模型的转写质量。
3. 用 `Qwen3-VL-2B` 重跑全部实验，记录质量差异——什么任务小模型就够了？
4. 视频理解（需较多显存）：给 `chat` 的 content 换成 `{"type": "video", "video": "path.mp4"}` 试试视频问答。

**下一站**：[02_api_frontier.ipynb](02_api_frontier.ipynb) — 同一组任务喂给三家闭源前沿 API。